# LeetCode 3263 — Convert Doubly Linked List to Array I bằng Python

Link đề: https://leetcode.com/problems/convert-doubly-linked-list-to-array-i/ · Độ khó: **Easy** · Chủ đề: Array, Linked List, Doubly-Linked List (danh sách liên kết đôi)

> **Bài Premium (🔒).** Đề trong notebook này được viết theo hiểu biết về bài (kèm các ví dụ của trang đề), **không** chép từ trang Premium. Nếu bạn có tài khoản Premium, hãy **đối chiếu lại đề gốc** (nhất là skeleton, ví dụ và ràng buộc) trước khi nộp.

Skeleton LeetCode (LeetCode đã định nghĩa sẵn lớp `Node` trong chú thích — trong notebook này ta **tự dựng lớp `Node`** để chạy thử):

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val, prev=None, next=None):
        self.val = val
        self.prev = prev
        self.next = next
"""

class Solution:
    def toArray(self, root: 'Optional[Node]') -> List[int]:
        ...
```

Ghi chú: mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả kiểm tra tính toàn vẹn của liên kết `prev`/`next` và đối chiếu hàng nghìn danh sách ngẫu nhiên). Ba ví dụ chính lấy từ trang đề; những ví dụ tự dựng thêm có ghi rõ "tự dựng".

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho **nút đầu** `root` (head) của một **danh sách liên kết đôi** — mỗi nút có `val`, con trỏ `prev` (nút trước) và `next` (nút sau) — hãy trả về **mảng** chứa các giá trị **theo thứ tự** từ đầu đến cuối.
2. Nắm các công cụ Python: **lớp** (`class Node`, `__init__`, thuộc tính), **`None`**, **con trỏ là tham chiếu** (hai biến cùng trỏ một đối tượng), vòng **`while root:`** đi dọc danh sách, `list.append`, **đệ quy**, và `reversed` / `[::-1]` để đảo mảng.
3. Đi từ cách **đếm độ dài rồi đi `i` bước cho từng phần tử** (`O(n²)`) → **đi một lượt và `append`** (`O(n)`) → **bộ sinh (generator) các nút** → **đệ quy** → **đi tới đuôi rồi đi ngược bằng `prev`** để thấy vai trò của liên kết đôi.
4. Hiểu mấu chốt: duyệt danh sách liên kết là **"đi theo mũi tên `next` cho tới khi gặp `None`"**; bài này **không cần** `prev` (chỉ cần `next`), còn `prev` chỉ là "thông tin thừa" mà đề đưa vào để làm quen với cấu trúc.
5. Tránh các lỗi kinh điển: **quên `root = root.next`** (lặp vô hạn), dùng `while root.next:` làm **mất nút cuối**, append **nút** thay vì **giá trị `val`**, đi nhầm theo `prev` từ đầu (dừng ngay), và truy cập `root.val` khi `root` có thể là `None`.

## 2. Đề bài & ví dụ

Cho `root` là **nút đầu** của một **danh sách liên kết đôi**, trong đó mỗi nút có một con trỏ **`next`** (trỏ nút kế tiếp) và một con trỏ **`prev`** (trỏ nút đứng trước).

Hãy trả về một **mảng số nguyên** chứa các phần tử của danh sách liên kết **theo đúng thứ tự** (từ đầu tới cuối).

**Ví dụ 1**

- Input: `head = [1,2,3,4,3,2,1]`
- Output: `[1,2,3,4,3,2,1]`

**Ví dụ 2**

- Input: `head = [2,2,2,2,2]`
- Output: `[2,2,2,2,2]`

**Ví dụ 3**

- Input: `head = [3,2,3,2,3,2]`
- Output: `[3,2,3,2,3,2]`

(Ghi chú: ở LeetCode, `head = [...]` mô tả các giá trị của danh sách theo thứ tự; hệ thống tự dựng các nút nối `prev` / `next` rồi truyền nút đầu vào hàm. Bài chỉ là "đọc danh sách ra mảng" nên đầu ra trùng đầu vào. **Đối chiếu lại đề gốc** nếu bạn có Premium.)

**Ràng buộc:** số nút trong danh sách thuộc `[1, 50]`; `1 <= Node.val <= 50`.

In [1]:
from typing import List, Optional
import random


class Node:
    """Nút của danh sách liên kết đôi: giá trị val, nút trước prev, nút sau next."""

    def __init__(self, val, prev=None, next=None):
        self.val = val
        self.prev = prev
        self.next = next

    def __repr__(self):
        return f"Node({self.val})"


def build(values):
    """Dựng danh sách liên kết đôi từ list giá trị, trả về nút đầu (hoặc None nếu list rỗng)."""
    head = tail = None
    for v in values:
        node = Node(v)
        if head is None:
            head = tail = node
        else:
            tail.next = node
            node.prev = tail
            tail = node
    return head


examples = [
    ([1, 2, 3, 4, 3, 2, 1], [1, 2, 3, 4, 3, 2, 1]),
    ([2, 2, 2, 2, 2], [2, 2, 2, 2, 2]),
    ([3, 2, 3, 2, 3, 2], [3, 2, 3, 2, 3, 2]),
]
for values, expected in examples:
    print(f"head = {values} -> kỳ vọng {expected}")

head = [1, 2, 3, 4, 3, 2, 1] -> kỳ vọng [1, 2, 3, 4, 3, 2, 1]
head = [2, 2, 2, 2, 2] -> kỳ vọng [2, 2, 2, 2, 2]
head = [3, 2, 3, 2, 3, 2] -> kỳ vọng [3, 2, 3, 2, 3, 2]


## 3. Kiến thức nền

### 3.1. Lớp `Node` và thuộc tính

Một **lớp** (`class`) là bản thiết kế của đối tượng. `__init__` chạy khi tạo đối tượng và gán các **thuộc tính** qua `self`. Một nút của danh sách liên kết đôi gồm **ba** thuộc tính: `val` (dữ liệu), `prev` (nút trước), `next` (nút sau). Nút ở **đầu** có `prev = None`; nút ở **cuối** có `next = None`. `None` nghĩa là "không có gì / hết đường".

In [2]:
a = Node(10)
print(a.val, a.prev, a.next)     # 10 None None  -> nút lẻ loi, chưa nối với ai
a.val = 11
print(a.val, a)                  # 11 Node(11)

10 None None
11 Node(11)


### 3.2. Con trỏ là *tham chiếu*: nối hai nút

Trong Python, `a.next = b` nghĩa là "thuộc tính `next` của `a` **trỏ tới chính đối tượng `b`**" (không sao chép `b`). Để nối đôi ta nối **cả hai chiều**: `a.next = b` và `b.prev = a`. Khi đó đi từ `a` sang `b` bằng `a.next`, và quay lại bằng `b.prev`.

In [3]:
a, b, c = Node(1), Node(2), Node(3)
a.next = b
b.prev = a
b.next = c
c.prev = b
print("a.next.val   =", a.next.val)            # 2
print("c.prev.val   =", c.prev.val)            # 2
print("a.next is b  =", a.next is b)           # True: cùng một đối tượng
print("b.prev.val   =", b.prev.val, "| b.next.val =", b.next.val)   # 1 3
print("a.prev =", a.prev, "| c.next =", c.next)   # None None -> hai đầu mút

a.next.val   = 2
c.prev.val   = 2
a.next is b  = True
b.prev.val   = 1 | b.next.val = 3
a.prev = None | c.next = None


### 3.3. Hàm `build` ở trên: dựng danh sách từ list

`build(values)` tạo từng nút, giữ `tail` là nút cuối hiện tại; mỗi nút mới `node` được nối `tail.next = node` và `node.prev = tail`, rồi `tail = node`. Ta dùng nó để dựng đầu vào, vì LeetCode tự làm việc đó khi chấm.

In [4]:
head = build([5, 6, 7])
print(head, head.next, head.next.next)               # Node(5) Node(6) Node(7)
print(head.next.prev, head.next.next.prev)           # Node(5) Node(6)
print(build([]))                                     # None -> danh sách rỗng

Node(5) Node(6) Node(7)
Node(5) Node(6)
None


### 3.4. Vòng `while root:` — đi dọc danh sách

Một nút (đối tượng) được coi là **đúng** (truthy) trong điều kiện `if`/`while`, còn `None` là **sai** (falsy). Vì vậy `while root:` nghĩa là "chừng nào còn nút". Mỗi vòng ta **xử lý nút hiện tại** rồi **đi tiếp** bằng `root = root.next`. Khi `root` thành `None` thì dừng. Đây là khuôn mẫu duyệt cơ bản của mọi bài danh sách liên kết.

In [5]:
root = build([4, 5, 6])
while root:
    print("đang đứng ở nút", root.val)
    root = root.next
print("kết thúc, root =", root)       # None

đang đứng ở nút 4
đang đứng ở nút 5
đang đứng ở nút 6
kết thúc, root = None


### 3.5. `list.append` và danh sách kết quả

`ans = []` tạo list rỗng, `ans.append(x)` thêm `x` vào **cuối** (`O(1)` khấu hao). Sau khi duyệt xong ta trả `ans`. Chú ý **append giá trị `root.val`** (số), không phải `root` (nút).

In [6]:
ans = []
root = build([7, 8, 9])
while root:
    ans.append(root.val)
    root = root.next
print(ans)                             # [7, 8, 9]

[7, 8, 9]


### 3.6. Đệ quy trên danh sách liên kết

Danh sách là cấu trúc **tự tương tự**: danh sách = nút đầu + (danh sách ngắn hơn bắt đầu từ `next`). Do đó có thể viết đệ quy: giá trị của nút đầu, rồi nối với kết quả của phần còn lại; **trường hợp dừng** là `None` (danh sách rỗng) trả `[]`. Với tối đa `50` nút, độ sâu đệ quy nhỏ, không lo tràn ngăn xếp.

In [7]:
def dem(root):
    if root is None:            # trường hợp cơ sở: danh sách rỗng
        return 0
    return 1 + dem(root.next)    # 1 nút đầu + số nút phần còn lại

print(dem(build([1, 2, 3, 4])), dem(None))     # 4 0

4 0


### 3.7. `reversed` và `[::-1]` để đảo list

`lst[::-1]` tạo list đảo ngược (bản sao); `list(reversed(lst))` cũng vậy; `lst.reverse()` đảo **tại chỗ**. Ta sẽ dùng khi đi từ đuôi về đầu bằng `prev`.

In [8]:
lst = [1, 2, 3]
print(lst[::-1], list(reversed(lst)))     # [3, 2, 1] [3, 2, 1]
lst.reverse()
print(lst)                                 # [3, 2, 1]

[3, 2, 1] [3, 2, 1]
[3, 2, 1]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Vẽ danh sách `[1, 2, 3]`:

```
None <- [1] <-> [2] <-> [3] -> None
        root
```

Muốn viết ra mảng ta làm đúng như đọc một dãy toa tàu:

1. Đứng ở toa đầu (`root`, giá trị `1`) → ghi `1`.
2. Đi sang toa sau theo `next` (giá trị `2`) → ghi `2`.
3. Đi tiếp (giá trị `3`) → ghi `3`.
4. Đi tiếp: `next` là `None` → **hết tàu**, dừng.

Kết quả `[1, 2, 3]`. Con trỏ `prev` **không dùng** trong cách đọc này — ta chỉ đi **về phía trước**.

Tương tự, nếu đứng ở toa cuối ta có thể đọc **ngược** bằng `prev`, rồi đảo mảng — đó là cách dùng `prev` (Cách 5).

In [9]:
head = build([1, 2, 3])
step = head
ghi = []
while step is not None:
    print(f"đứng ở {step.val}: ghi {step.val}, đi tiếp tới", step.next)
    ghi.append(step.val)
    step = step.next
print(ghi)

đứng ở 1: ghi 1, đi tiếp tới Node(2)
đứng ở 2: ghi 2, đi tiếp tới Node(3)
đứng ở 3: ghi 3, đi tiếp tới None
[1, 2, 3]


## 5. Cách 1 — Brute Force / baseline: đếm độ dài rồi "đi `i` bước" cho từng phần tử

**Ý tưởng.** Giống như tra mảng theo chỉ số: biết độ dài `n`, với mỗi `i` từ `0` đến `n - 1` ta **xuất phát lại từ đầu**, đi `i` bước `next` để tới nút thứ `i` và lấy `val`. Rất "ngây thơ" vì mỗi phần tử phải đi lại từ đầu.

In [10]:
def to_array_brute(root):
    # Bước 1: đếm độ dài
    n = 0
    cur = root
    while cur:
        n += 1
        cur = cur.next
    # Bước 2: với mỗi chỉ số i, đi từ đầu i bước
    ans = []
    for i in range(n):
        cur = root
        for _ in range(i):
            cur = cur.next
        ans.append(cur.val)
    return ans


for values, expected in examples:
    print(values, "->", to_array_brute(build(values)), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 3, 2, 1] -> [1, 2, 3, 4, 3, 2, 1] (kỳ vọng [1, 2, 3, 4, 3, 2, 1] )
[2, 2, 2, 2, 2] -> [2, 2, 2, 2, 2] (kỳ vọng [2, 2, 2, 2, 2] )
[3, 2, 3, 2, 3, 2] -> [3, 2, 3, 2, 3, 2] (kỳ vọng [3, 2, 3, 2, 3, 2] )


**Độ phức tạp.** Đếm `O(n)`; phần xây mảng: phần tử thứ `i` tốn `i` bước ⇒ tổng `0 + 1 + ... + (n−1) = n(n−1)/2 = O(n²)`. Space `O(1)` ngoài mảng kết quả.

**Nút thắt.** Mỗi lần ta **quên** mình đang đứng ở đâu và đi lại từ đầu. Chỉ cần **giữ vị trí hiện tại** và đi tiếp một bước là đủ — đó là ý tưởng của Cách 2.

## 6. Ý tưởng thuật toán / bước đột phá

Danh sách liên kết không cho **truy cập ngẫu nhiên** (không có `list[i]` tức thời) nhưng cho **đi tiếp rất rẻ**: từ nút hiện tại sang nút sau là `O(1)`. Vì thế đừng quay về đầu — hãy **đi một lượt duy nhất**:

> giữ một con trỏ `cur` bắt đầu ở `root`; mỗi bước **ghi `cur.val`** rồi **`cur = cur.next`**; dừng khi `cur` là `None`.

Mỗi nút được thăm **đúng một lần** ⇒ `O(n)`. Con trỏ `prev` không cần thiết cho yêu cầu "đọc theo thứ tự" nên ta bỏ qua nó (nhưng sẽ dùng thử ở Cách 5).

In [11]:
# Đo "số bước đi" của hai cách trên cùng danh sách để thấy khác biệt
def dem_buoc_brute(n):
    return n + sum(range(n))          # đếm n bước + tổng 0..n-1 bước đi lại


def dem_buoc_mot_luot(n):
    return n                          # mỗi nút một bước


for n in [1, 5, 10, 50]:
    print(f"n={n}: vét cạn ~{dem_buoc_brute(n)} bước, một lượt {dem_buoc_mot_luot(n)} bước")

n=1: vét cạn ~1 bước, một lượt 1 bước
n=5: vét cạn ~15 bước, một lượt 5 bước
n=10: vét cạn ~55 bước, một lượt 10 bước
n=50: vét cạn ~1275 bước, một lượt 50 bước


## 7. Các cách tiếp theo

Phần này có **4 cách** nữa (cùng Cách 1 là 5 cách). Mỗi cách: giải thích → code demo → code nộp LeetCode → độ phức tạp.

### Cách 2 — Duyệt một lượt bằng `while` + `append` (cách chuẩn)

Giải thích từng đoạn:

- `ans = []`: nơi gom giá trị.
- `while root:`: còn nút thì còn xử lý (xem 3.4).
- `ans.append(root.val)`: ghi **giá trị** của nút hiện tại.
- `root = root.next`: **đi tiếp** — thiếu dòng này là lặp vô hạn.
- Hết nút thì `root` là `None`, thoát vòng, trả `ans`.

In [12]:
def to_array_while(root):
    ans = []
    while root:
        ans.append(root.val)
        root = root.next
    return ans


for values, expected in examples:
    assert to_array_while(build(values)) == expected
print("Cách 2 chạy đúng trên các ví dụ")

Cách 2 chạy đúng trên các ví dụ


In [13]:
# Code nộp LeetCode — Cách 2
class Solution:
    def toArray(self, root: 'Optional[Node]') -> List[int]:
        ans = []
        while root:
            ans.append(root.val)
            root = root.next
        return ans


for values, expected in examples:
    assert Solution().toArray(build(values)) == expected
print("OK")

OK


**Độ phức tạp Cách 2:** Time `O(n)`, Space `O(1)` ngoài mảng kết quả (mảng kết quả `O(n)` là bắt buộc vì phải trả về).

### Cách 3 — Bộ sinh (generator) các nút + list comprehension

Giải thích từng đoạn:

- `iter_nodes(root)` là **generator**: hàm có `yield`, mỗi lần `yield` "nhả" một giá trị rồi dừng chờ lần lấy tiếp theo.
- `yield cur` trả về **nút**; khi dùng `for node in iter_nodes(root)` ta lấy được từng nút theo thứ tự.
- `[node.val for node in iter_nodes(root)]` gom giá trị thành mảng. Cách này **tách** "đi dọc danh sách" khỏi "làm gì với từng nút", rất tiện cho các bài sau (tính tổng, tìm phần tử...).

In [14]:
def iter_nodes(root):
    cur = root
    while cur:
        yield cur
        cur = cur.next


def to_array_gen(root):
    return [node.val for node in iter_nodes(root)]


for values, expected in examples:
    assert to_array_gen(build(values)) == expected
print("Cách 3 chạy đúng trên các ví dụ")
print(list(iter_nodes(build([4, 5]))))     # [Node(4), Node(5)]

Cách 3 chạy đúng trên các ví dụ
[Node(4), Node(5)]


In [15]:
# Code nộp LeetCode — Cách 3
class Solution:
    def toArray(self, root: 'Optional[Node]') -> List[int]:
        def iter_nodes(cur):
            while cur:
                yield cur
                cur = cur.next

        return [node.val for node in iter_nodes(root)]


for values, expected in examples:
    assert Solution().toArray(build(values)) == expected
print("OK")

OK


**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(1)` ngoài mảng kết quả (generator chỉ giữ một nút tại một thời điểm).

### Cách 4 — Đệ quy

Giải thích từng đoạn:

- **Trường hợp cơ sở:** `root is None` → trả `[]`.
- **Bước đệ quy:** `[root.val] + to_array_rec(root.next)` = giá trị của nút này, rồi tới mảng của phần còn lại.
- Mỗi nút gọi đệ quy một lần ⇒ `n` lệnh gọi, độ sâu `n`. (Phép nối list tạo bản sao nên tổng chi phí là `O(n²)` lý thuyết; với `n <= 50` không đáng kể. Cách tránh: truyền một list tích lũy `acc` và chỉ `append`.)

In [16]:
def to_array_rec(root):
    if root is None:
        return []
    return [root.val] + to_array_rec(root.next)


def to_array_rec_acc(root, acc=None):
    """Bản đệ quy dùng list tích lũy: mỗi nút chỉ append một lần -> O(n)."""
    if acc is None:
        acc = []
    if root is None:
        return acc
    acc.append(root.val)
    return to_array_rec_acc(root.next, acc)


for values, expected in examples:
    assert to_array_rec(build(values)) == expected
    assert to_array_rec_acc(build(values)) == expected
print("Cách 4 chạy đúng trên các ví dụ")

Cách 4 chạy đúng trên các ví dụ


In [17]:
# Code nộp LeetCode — Cách 4
class Solution:
    def toArray(self, root: 'Optional[Node]') -> List[int]:
        if root is None:
            return []
        return [root.val] + self.toArray(root.next)


for values, expected in examples:
    assert Solution().toArray(build(values)) == expected
print("OK")

OK


**Độ phức tạp Cách 4:** Time `O(n²)` với phép nối list (hoặc `O(n)` với bản tích lũy), Space `O(n)` cho ngăn xếp đệ quy. Với `n <= 50` mọi thứ đều nhỏ.

### Cách 5 — Đi tới đuôi rồi đi ngược bằng `prev`, rồi đảo mảng

Giải thích từng đoạn: đây là cách **dùng đến liên kết `prev`**, để thấy vì sao danh sách là "liên kết đôi".

- Đi tới đuôi: `tail = root; while tail.next: tail = tail.next` (dừng ở nút **cuối cùng**, chưa tới `None`).
- Từ đuôi đi ngược: `while tail: ans.append(tail.val); tail = tail.prev`. Thu được mảng **ngược** `[a_n, ..., a_1]`.
- `ans.reverse()` (hoặc `[::-1]`) để có thứ tự đúng.
- Cần xử lý trường hợp `root` là `None` ngay từ đầu (không có `tail`).

In [18]:
def to_array_backward(root):
    if root is None:
        return []
    tail = root
    while tail.next:                    # đi tới nút cuối (next là None)
        tail = tail.next
    ans = []
    while tail:                         # từ đuôi đi ngược theo prev
        ans.append(tail.val)
        tail = tail.prev
    ans.reverse()
    return ans


for values, expected in examples:
    assert to_array_backward(build(values)) == expected
print("Cách 5 chạy đúng trên các ví dụ")

Cách 5 chạy đúng trên các ví dụ


In [19]:
# Code nộp LeetCode — Cách 5
class Solution:
    def toArray(self, root: 'Optional[Node]') -> List[int]:
        if root is None:
            return []
        tail = root
        while tail.next:
            tail = tail.next
        ans = []
        while tail:
            ans.append(tail.val)
            tail = tail.prev
        ans.reverse()
        return ans


for values, expected in examples:
    assert Solution().toArray(build(values)) == expected
print("OK")

OK


**Độ phức tạp Cách 5:** Time `O(n)` (đi tới đuôi `n` bước, đi ngược `n` bước, đảo `n`), Space `O(1)` ngoài mảng. Chậm hơn Cách 2 khoảng `3` lần (hằng số) và **không cần thiết** ở bài này — nhưng cho thấy vì sao người ta thiết kế `prev`: khi đã có một nút giữa danh sách hoặc đuôi, ta vẫn đi ngược được.

## 8. TRACE từng bước

Ta trace Cách 2 (duyệt một lượt) trên Ví dụ 1; Cách 5 (đi tới đuôi rồi đi ngược) trên `[3, 1, 4]` (tự dựng) để thấy `prev` hoạt động; và Cách 4 (đệ quy) trên `[7, 8, 9]` để thấy các lệnh gọi lồng nhau.

In [20]:
def trace_while(root):
    print("danh sách:", to_array_while(root))
    ans = []
    step = 0
    while root:
        step += 1
        ans.append(root.val)
        prev_val = root.prev.val if root.prev else None
        next_val = root.next.val if root.next else None
        print(f"  bước {step}: root.val={root.val} (prev={prev_val}, next={next_val}) -> ans = {ans}")
        root = root.next
    print(f"  root = {root} -> dừng")
    return ans


assert trace_while(build([1, 2, 3, 4, 3, 2, 1])) == [1, 2, 3, 4, 3, 2, 1]

danh sách: [1, 2, 3, 4, 3, 2, 1]
  bước 1: root.val=1 (prev=None, next=2) -> ans = [1]
  bước 2: root.val=2 (prev=1, next=3) -> ans = [1, 2]
  bước 3: root.val=3 (prev=2, next=4) -> ans = [1, 2, 3]
  bước 4: root.val=4 (prev=3, next=3) -> ans = [1, 2, 3, 4]
  bước 5: root.val=3 (prev=4, next=2) -> ans = [1, 2, 3, 4, 3]
  bước 6: root.val=2 (prev=3, next=1) -> ans = [1, 2, 3, 4, 3, 2]
  bước 7: root.val=1 (prev=2, next=None) -> ans = [1, 2, 3, 4, 3, 2, 1]
  root = None -> dừng


In [21]:
def trace_backward(root):
    print("danh sách:", to_array_while(root))
    tail = root
    hops = 0
    while tail.next:
        tail = tail.next
        hops += 1
        print(f"  đi tới đuôi: nhảy {hops} bước, đang ở {tail.val}")
    print(f"  đuôi = {tail.val}")
    ans = []
    while tail:
        ans.append(tail.val)
        print(f"  đi ngược: ghi {tail.val}, ans (ngược) = {ans}")
        tail = tail.prev
    ans.reverse()
    print("  sau đảo:", ans)
    return ans


assert trace_backward(build([3, 1, 4])) == [3, 1, 4]

danh sách: [3, 1, 4]
  đi tới đuôi: nhảy 1 bước, đang ở 1
  đi tới đuôi: nhảy 2 bước, đang ở 4
  đuôi = 4
  đi ngược: ghi 4, ans (ngược) = [4]
  đi ngược: ghi 1, ans (ngược) = [4, 1]
  đi ngược: ghi 3, ans (ngược) = [4, 1, 3]
  sau đảo: [3, 1, 4]


In [22]:
def trace_rec(root, depth=0):
    pad = "  " * (depth + 1)
    if root is None:
        print(f"{pad}to_array_rec(None) -> []")
        return []
    print(f"{pad}to_array_rec({root.val}) gọi đệ quy cho phần còn lại ...")
    rest = trace_rec(root.next, depth + 1)
    out = [root.val] + rest
    print(f"{pad}to_array_rec({root.val}) = [{root.val}] + {rest} = {out}")
    return out


assert trace_rec(build([7, 8, 9])) == [7, 8, 9]

  to_array_rec(7) gọi đệ quy cho phần còn lại ...
    to_array_rec(8) gọi đệ quy cho phần còn lại ...
      to_array_rec(9) gọi đệ quy cho phần còn lại ...
        to_array_rec(None) -> []
      to_array_rec(9) = [9] + [] = [9]
    to_array_rec(8) = [8] + [9] = [8, 9]
  to_array_rec(7) = [7] + [8, 9] = [7, 8, 9]


Bảng trace Cách 2 với Ví dụ 1 (`[1, 2, 3, 4, 3, 2, 1]`):

| bước | `root.val` | `prev` | `next` | `ans` sau bước |
|---|---|---|---|---|
| 1 | 1 | None | 2 | [1] |
| 2 | 2 | 1 | 3 | [1, 2] |
| 3 | 3 | 2 | 4 | [1, 2, 3] |
| 4 | 4 | 3 | 3 | [1, 2, 3, 4] |
| 5 | 3 | 4 | 2 | [1, 2, 3, 4, 3] |
| 6 | 2 | 3 | 1 | [1, 2, 3, 4, 3, 2] |
| 7 | 1 | 2 | None | [1, 2, 3, 4, 3, 2, 1] |
| (hết) | `root = None` | | | dừng, trả `ans` |

Để ý hai điều: (1) `prev` ở mỗi hàng đúng bằng `root.val` của hàng trước — đó là tính chất của danh sách liên kết đôi; (2) giá trị lặp lại (`3`, `2`, `1`) không gây vấn đề vì ta đi theo **nút**, không theo giá trị.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 2** (duyệt một lượt): ngắn nhất, nhanh nhất và không có điểm lạ.

In [23]:
class Solution:
    def toArray(self, root: 'Optional[Node]') -> List[int]:
        ans = []                       # kết quả
        while root:                    # còn nút thì còn xử lý
            ans.append(root.val)       # ghi giá trị (không phải nút)
            root = root.next           # đi tiếp; hết nút thì root là None
        return ans


sol = Solution()
for values, expected in examples:
    print(values, "->", sol.toArray(build(values)), "(kỳ vọng", expected, ")")

[1, 2, 3, 4, 3, 2, 1] -> [1, 2, 3, 4, 3, 2, 1] (kỳ vọng [1, 2, 3, 4, 3, 2, 1] )
[2, 2, 2, 2, 2] -> [2, 2, 2, 2, 2] (kỳ vọng [2, 2, 2, 2, 2] )
[3, 2, 3, 2, 3, 2] -> [3, 2, 3, 2, 3, 2] (kỳ vọng [3, 2, 3, 2, 3, 2] )


## 10. Vì sao thuật toán đúng

- **Bất biến vòng lặp.** Trước mỗi vòng `while`, `ans` chứa **đúng** các giá trị của những nút **đã đi qua** (theo thứ tự), và `root` là nút **kế tiếp chưa ghi** (hoặc `None` nếu đã hết). Ban đầu `ans = []`, `root` là nút đầu: đúng.
- **Duy trì bất biến.** Mỗi vòng ghi `root.val` rồi chuyển sang `root.next`: `ans` thêm đúng nút kế, `root` thành nút kế tiếp nữa.
- **Dừng và đúng thứ tự.** Danh sách hữu hạn, mỗi vòng tiến một nút nên sau tối đa `n` vòng `root` thành `None` (nút cuối có `next = None`). Lúc đó `ans` chứa mọi giá trị theo thứ tự từ đầu tới cuối.
- **Không đụng `prev`.** Vì chỉ đọc `val` và `next`, ta **không sửa** cấu trúc danh sách; liên kết `prev`/`next` sau hàm vẫn nguyên (ta kiểm tra điều này bằng test ở mục 12).
- **Đối chiếu độc lập.** Cách 1 (đi `i` bước theo chỉ số) và Cách 5 (đi ngược bằng `prev`) dùng đường đi khác hẳn nhưng cho cùng mảng — nếu liên kết đúng, hai đường phải khớp nhau.

## 11. Time / Space Complexity

Gọi `n` là số nút.

| Cách | Time | Space (thêm, ngoài mảng kết quả) |
|---|---|---|
| 1. Đếm độ dài rồi đi `i` bước | `O(n²)` | `O(1)` |
| 2. `while` + `append` | `O(n)` | `O(1)` |
| 3. Generator + list comprehension | `O(n)` | `O(1)` |
| 4. Đệ quy | `O(n²)` (nối list) / `O(n)` (tích lũy) | `O(n)` (ngăn xếp) |
| 5. Tới đuôi rồi đi ngược `prev` | `O(n)` | `O(1)` |

Mảng kết quả `O(n)` là bắt buộc nên Time `O(n)` là tối ưu (phải đọc mọi nút). Với `n <= 50` mọi cách đều chạy trong chớp mắt.

## 12. Bộ test

Gồm: ví dụ đề, biên (một nút, hai nút, toàn bộ giá trị bằng nhau, đủ `50` nút với giá trị lớn nhất `50`), kiểm tra **liên kết không bị hỏng** sau khi gọi hàm, đối chiếu cả `5` cách và ngẫu nhiên có seed đúng ràng buộc (`1 <= n <= 50`, `1 <= val <= 50`).

In [24]:
def verify_links(head):
    """Kiểm tra tính toàn vẹn của danh sách liên kết đôi: prev của đầu là None, và với mọi nút
    node.next.prev is node. Trả về danh sách nút theo chiều xuôi."""
    nodes = []
    cur = head
    while cur:
        nodes.append(cur)
        cur = cur.next
    if nodes:
        assert nodes[0].prev is None
        assert nodes[-1].next is None
    for a, b in zip(nodes, nodes[1:]):
        assert a.next is b and b.prev is a
    return nodes


IMPLS = {
    "brute": to_array_brute,
    "while": to_array_while,
    "gen": to_array_gen,
    "rec": to_array_rec,
    "rec_acc": to_array_rec_acc,
    "backward": to_array_backward,
    "Solution": Solution().toArray,
}

# 1) ví dụ đề
for values, expected in examples:
    for name, f in IMPLS.items():
        head = build(values)
        assert f(head) == expected, (name, values)
        verify_links(head)                       # liên kết còn nguyên sau khi gọi hàm

# 2) biên và tự dựng
manual = [
    [1],                                         # một nút
    [50],                                        # giá trị lớn nhất
    [1, 2],                                      # hai nút
    [7] * 50,                                    # 50 nút toàn bằng nhau
    list(range(1, 51)),                          # tăng dần đủ 50 nút
    list(range(50, 0, -1)),                      # giảm dần đủ 50 nút
    [3, 3, 1, 1, 3, 3],                          # lặp theo cặp
]
for values in manual:
    for name, f in IMPLS.items():
        head = build(values)
        assert f(head) == values, (name, values)
        assert [n.val for n in verify_links(head)] == values
print("ví dụ đề + biên: PASS")

ví dụ đề + biên: PASS


In [25]:
# 3) ngẫu nhiên có seed: đúng ràng buộc, mọi cách phải trả đúng mảng gốc
rng = random.Random(3263)
for _ in range(3000):
    n = rng.randint(1, 50)
    values = [rng.randint(1, 50) for _ in range(n)]
    for name, f in IMPLS.items():
        head = build(values)
        assert f(head) == values, (name, values)
    # kiểm tra liên kết còn nguyên sau khi chạy nhiều cách trên cùng một danh sách
    head = build(values)
    for f in IMPLS.values():
        f(head)
    assert [n.val for n in verify_links(head)] == values
print("3000 danh sách ngẫu nhiên: tất cả các cách đồng nhất — PASS")

# 4) bản sao độc lập: sửa mảng trả về không ảnh hưởng danh sách
head = build([1, 2, 3])
arr = Solution().toArray(head)
arr[0] = 99
assert head.val == 1 and Solution().toArray(head) == [1, 2, 3]
print("mảng trả về độc lập với danh sách: PASS")

3000 danh sách ngẫu nhiên: tất cả các cách đồng nhất — PASS
mảng trả về độc lập với danh sách: PASS


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `root = root.next` (lặp vô hạn)

Nếu không đi tiếp, `root` mãi là nút đầu: điều kiện `while root:` luôn đúng, `ans` phình ra mãi. Dưới đây ta đặt **giới hạn an toàn** để minh họa mà không treo máy.

In [26]:
def sai_quen_di_tiep(root, limit=8):
    ans = []
    while root:
        ans.append(root.val)
        # SAI: thiếu root = root.next
        if len(ans) >= limit:          # chỉ để thoát khi minh họa (đề thật sẽ treo)
            break
    return ans

print("sai:", sai_quen_di_tiep(build([1, 2, 3])))       # [1, 1, 1, 1, 1, 1, 1, 1]
print("đúng:", to_array_while(build([1, 2, 3])))

sai: [1, 1, 1, 1, 1, 1, 1, 1]
đúng: [1, 2, 3]


### Lỗi 2: Dùng `while root.next:` — mất nút cuối

`root.next` là `None` tại nút cuối, nên vòng dừng **trước khi** ghi nút cuối. (Nó cũng báo lỗi nếu `root` là `None` vì `None.next` không tồn tại.)

In [27]:
def sai_mat_nut_cuoi(root):
    ans = []
    while root.next:                  # SAI: dừng khi đang đứng ở nút cuối
        ans.append(root.val)
        root = root.next
    return ans

print("sai:", sai_mat_nut_cuoi(build([1, 2, 3])), "(thiếu 3)")
print("đúng:", to_array_while(build([1, 2, 3])))

sai: [1, 2] (thiếu 3)
đúng: [1, 2, 3]


### Lỗi 3: Append **nút** thay vì **giá trị**

`ans.append(root)` đưa **đối tượng** `Node` vào mảng. Kiểu trả về sai (đề yêu cầu số nguyên), và so sánh `==` với `[1, 2, 3]` cho `False`.

In [28]:
def sai_append_nut(root):
    ans = []
    while root:
        ans.append(root)              # SAI: phải là root.val
        root = root.next
    return ans

res = sai_append_nut(build([1, 2, 3]))
print("sai:", res, "| bằng [1, 2, 3]?", res == [1, 2, 3])
print("đúng:", to_array_while(build([1, 2, 3])))

sai: [Node(1), Node(2), Node(3)] | bằng [1, 2, 3]? False
đúng: [1, 2, 3]


### Lỗi 4: Đi nhầm theo `prev` từ nút đầu

Nút đầu có `prev = None` nên đi `root = root.prev` dừng ngay sau **một** nút. Muốn đi tới các nút sau từ đầu phải dùng `next`; `prev` chỉ dùng khi đi **ngược** từ đuôi.

In [29]:
def sai_di_prev(root):
    ans = []
    while root:
        ans.append(root.val)
        root = root.prev              # SAI: từ đầu thì prev là None
    return ans

print("sai:", sai_di_prev(build([1, 2, 3])))    # [1]
print("đúng:", to_array_while(build([1, 2, 3])))

sai: [1]
đúng: [1, 2, 3]


### Lỗi 5: Truy cập `.val` khi `root` có thể là `None`

Ở bài này danh sách có ít nhất `1` nút, nhưng thói quen tốt là **kiểm tra `None` trước khi dùng `.val` / `.next`** (các bài sau thường có danh sách rỗng). Viết `while root:` đã bao hàm kiểm tra này.

In [30]:
def sai_khong_kiem_tra_none(root):
    ans = [root.val]                  # SAI nếu root là None: AttributeError
    while root.next:
        root = root.next
        ans.append(root.val)
    return ans

try:
    sai_khong_kiem_tra_none(None)
except AttributeError as e:
    print("AttributeError:", e)
print("đúng (không lỗi):", to_array_while(None))      # []

AttributeError: 'NoneType' object has no attribute 'val'
đúng (không lỗi): []


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Mẫu duyệt danh sách liên kết:** `while cur: ...; cur = cur.next`. Nếu cần nút cuối thì `while cur.next:` (dừng **ở** nút cuối). Phân biệt hai điều kiện này là kỹ năng cốt lõi.
- **Danh sách liên kết đôi:** nhớ hai bất biến `a.next.prev is a` và `b.prev.next is b`; đầu có `prev = None`, đuôi có `next = None`. Khi **sửa** danh sách (chèn/xóa) phải cập nhật **cả hai chiều**.
- **Không có truy cập ngẫu nhiên:** đừng viết `for i in range(n): đi i bước` (`O(n²)`). Hãy giữ con trỏ và đi tiếp.
- **Hầu hết bài đọc/đếm/tìm trên danh sách** là một lượt duyệt `O(n)`; bài khó mới cần hai con trỏ (nhanh/chậm), đảo ngược, hoặc nút giả (dummy).
- **Chuyển mảng ⇄ danh sách để tự test:** luôn có hàm `build(values)` và hàm `to_array(head)`; kiểm tra thêm bằng hàm `verify_links` để bắt lỗi nối thiếu một chiều.
- **Đừng sửa đầu vào** nếu đề không yêu cầu: bài này chỉ đọc nên không đụng `prev`/`next`.

## 15. Biến thể của bài

1. **Đảo ngược mảng:** trả về các giá trị theo thứ tự **từ cuối về đầu** — duyệt xuôi rồi `reverse`, hoặc đi tới đuôi rồi dùng `prev` (không cần đảo).
2. **Bắt đầu từ nút bất kỳ (bài II):** cho **một nút ở giữa** danh sách (không phải đầu), trả về **toàn bộ** danh sách dưới dạng mảng. Cần đi **lùi** bằng `prev` tới đầu rồi đi xuôi — đây chính là lúc `prev` có ích.
3. **Độ dài / tổng / lớn nhất** thay vì mảng: cùng một lượt duyệt, chỉ khác phép tích lũy.
4. **Danh sách vòng (circular):** nút cuối trỏ lại nút đầu; cần dừng khi quay lại đầu thay vì chờ `None`.
5. **Xóa/chèn nút** trong danh sách liên kết đôi và cập nhật cả `prev` lẫn `next`.

In [31]:
# Minh họa biến thể 1 (đảo ngược) và biến thể 2 (từ một nút giữa trả về toàn bộ danh sách)
def to_array_reversed(root):
    return to_array_while(root)[::-1]


def whole_list_from_any_node(node):
    while node.prev:                  # lùi về đầu bằng prev
        node = node.prev
    ans = []
    while node:
        ans.append(node.val)
        node = node.next
    return ans


head = build([4, 5, 6, 7])
mid = head.next.next                  # nút có giá trị 6
print(to_array_reversed(head))        # [7, 6, 5, 4]
print(whole_list_from_any_node(mid))  # [4, 5, 6, 7]

[7, 6, 5, 4]
[4, 5, 6, 7]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space (thêm) | Khi nào dùng |
|---|---|---|---|---|
| 1. Đếm rồi đi `i` bước | Mô phỏng truy cập chỉ số | `O(n²)` | `O(1)` | Chỉ để minh họa vì sao **không** nên làm vậy |
| 2. `while` + `append` | Đi một lượt theo `next` | `O(n)` | `O(1)` | **Cách nộp chuẩn** |
| 3. Generator | Tách "đi dọc" khỏi "xử lý" | `O(n)` | `O(1)` | Khi nhiều bài cùng cần duyệt; code gọn |
| 4. Đệ quy | Nút đầu + phần còn lại | `O(n²)` / `O(n)` | `O(n)` | Luyện tư duy đệ quy; danh sách ngắn |
| 5. Tới đuôi, đi ngược `prev` | Dùng liên kết `prev` | `O(n)` | `O(1)` | Khi chỉ có đuôi/nút giữa; hiểu danh sách đôi |

Lời khuyên: nộp **Cách 2**; nhớ **Cách 5** vì nó là nền cho bài "bắt đầu từ nút bất kỳ".

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `build([5, 6, 7]).next.prev.val`, `build([5, 6, 7]).prev`, `Solution().toArray(build([9]))`, `Solution().toArray(build([1, 1, 2]))`.

**Bài 2 (tính tay).** Cho danh sách `[2, 4, 6, 8]` có nút đầu `h`. Viết ra giá trị của `h.next.next.val`, `h.next.next.next.prev.val` và `h.next.next.next.next`. Vẽ sơ đồ `None <-> ... <-> None` để kiểm tra.

**Bài 3 (điền TODO).** Hoàn thành `to_array_todo(root)` theo Cách 2.

In [32]:
def to_array_todo(root):
    # TODO 1: tạo list rỗng ans
    # TODO 2: while root: ghi root.val vào ans
    # TODO 3: đi tiếp root = root.next
    # TODO 4: trả về ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Không nhìn lời giải, viết `to_array_from_tail(tail)` nhận **nút cuối** của danh sách và trả về mảng theo thứ tự **từ đầu tới cuối** chỉ dùng `prev` (gợi ý: đi ngược rồi `reverse`, hoặc chèn vào đầu).

**Bài 5 (biến thể / mini challenge).** (a) Viết `length(root)` đếm số nút. (b) Viết `whole_array_from_middle(node)` nhận **một nút bất kỳ** và trả về mảng toàn bộ danh sách. (c) Viết `remove_node(node)` xóa một nút **không phải đầu cũng không phải đuôi** khỏi danh sách (cập nhật cả `prev` lẫn `next` của hai hàng xóm) rồi dùng `verify_links` để kiểm tra danh sách còn nối đúng.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `build([5,6,7])` là `5 <-> 6 <-> 7`; `next.prev` quay lại chính nút `5`; nút đầu có `prev` là `None`. Bài 2: mỗi `.next` đi một nút sang phải, mỗi `.prev` đi một nút sang trái; sau nút cuối là `None`.
- **Mức 2 (vừa):** Bài 3: giống mục 9. Bài 4: `tail` đi ngược bằng `tail = tail.prev`, ghi `val` vào list rồi `reverse()`.
- **Mức 3 (sâu):** Bài 5(b): lùi bằng `while node.prev: node = node.prev` rồi duyệt xuôi. Bài 5(c): với `p = node.prev`, `q = node.next`: `p.next = q` và `q.prev = p` (cả hai chiều!); có thể thêm `node.prev = node.next = None` để "cắt" hẳn nút.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert build([5, 6, 7]).next.prev.val == 5
assert build([5, 6, 7]).prev is None
assert Solution().toArray(build([9])) == [9]
assert Solution().toArray(build([1, 1, 2])) == [1, 1, 2]

# Bài 2: [2, 4, 6, 8]
h = build([2, 4, 6, 8])
assert h.next.next.val == 6
assert h.next.next.next.prev.val == 6          # nút 8 lùi về nút 6
assert h.next.next.next.next is None            # sau nút cuối là None


# Bài 3
def to_array_todo_ans(root):
    ans = []
    while root:
        ans.append(root.val)
        root = root.next
    return ans


for values, expected in examples:
    assert to_array_todo_ans(build(values)) == expected


# Bài 4
def to_array_from_tail(tail):
    ans = []
    while tail:
        ans.append(tail.val)
        tail = tail.prev
    ans.reverse()
    return ans


def last_node(head):
    while head.next:
        head = head.next
    return head


for values, expected in examples:
    assert to_array_from_tail(last_node(build(values))) == expected
assert to_array_from_tail(None) == []


# Bài 5
def length(root):
    n = 0
    while root:
        n += 1
        root = root.next
    return n


def whole_array_from_middle(node):
    while node.prev:
        node = node.prev
    ans = []
    while node:
        ans.append(node.val)
        node = node.next
    return ans


def remove_node(node):
    p, q = node.prev, node.next
    assert p is not None and q is not None      # chỉ xóa nút ở giữa
    p.next = q
    q.prev = p
    node.prev = node.next = None


assert length(build([1, 2, 3])) == 3 and length(None) == 0
head = build([10, 20, 30, 40, 50])
mid = head.next.next
assert whole_array_from_middle(mid) == [10, 20, 30, 40, 50]
remove_node(mid)
assert [n.val for n in verify_links(head)] == [10, 20, 40, 50]
assert Solution().toArray(head) == [10, 20, 40, 50]
assert to_array_from_tail(last_node(head)) == [10, 20, 40, 50]     # chiều ngược cũng khớp
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (duyệt danh sách liên kết, ghi giá trị mỗi nút)

```python
class Solution:
    def toArray(self, root: 'Optional[Node]') -> List[int]:
        ans = []
        while root:
            ans.append(root.val)
            root = root.next
        return ans
```

**Checklist**

- [ ] Có dòng **`root = root.next`** trong vòng lặp (không lặp vô hạn)?
- [ ] Điều kiện là `while root:` (không phải `while root.next:` làm mất nút cuối)?
- [ ] Append **`root.val`** (số), không append cả nút?
- [ ] Đi theo **`next`** từ đầu (không phải `prev`)?
- [ ] Không truy cập `.val` / `.next` khi `root` có thể là `None`?
- [ ] Đã thử ba ví dụ đề và biên (một nút, toàn giá trị bằng nhau)? Đã đối chiếu đề gốc nếu có tài khoản Premium?
- [ ] Skeleton đúng `class Solution` / `def toArray(self, root: 'Optional[Node]') -> List[int]`?